In [1]:
# Practical 9: Data Cleaning and Preparation
# Aim: Clean and prepare the customer churn data by handling missing values,
# duplicates, inconsistent values and outliers, and create a new feature.
# Dataset: Telecom_Customer_Churn.csv

import pandas as pd

df = pd.read_csv("telco.csv")
df.columns = df.columns.str.strip().str.replace(" ", "_")
print("Shape:", df.shape)

df = df.drop_duplicates()

for col in ["tenure", "MonthlyCharges", "TotalCharges"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

missing = df.isna().sum()
print("\nMissing values:\n", missing[missing > 0])

num_cols = df.select_dtypes(include="number").columns
text_cols = [c for c in df.columns if c not in num_cols and c != "customerID"]

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

for col in text_cols:
    df[col] = df[col].str.strip()
    df[col] = df[col].replace({"No internet service": "No", "No phone service": "No"})

df["Tenure_Group"] = pd.cut(df["tenure"], [0, 12, 24, 48, 72],
                            labels=["0-1 yr", "1-2 yr", "2-4 yr", "4-6 yr"], include_lowest=True)
df["Estimated_Lifetime_Charges"] = df["tenure"] * df["MonthlyCharges"]

print("\nOutliers (IQR method):")
for col in ["tenure", "MonthlyCharges", "TotalCharges"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    df[col + "_Outlier"] = (df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)
    print(col, ":", df[col + "_Outlier"].sum())

print("\nShape after cleaning:", df.shape)
print("Missing values left:", df.isna().sum().sum())
df.to_csv("Telecom_Customer_Churn_Cleaned.csv", index=False)

Shape: (7043, 21)

Missing values:
 TotalCharges    11
dtype: int64

Outliers (IQR method):
tenure : 0
MonthlyCharges : 0
TotalCharges : 0

Shape after cleaning: (7043, 26)
Missing values left: 0
